# Diagnosing and Handling Weak Instruments

**Module 06 | Estimated time: 15 minutes**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Show empirically how weak instruments bias 2SLS towards OLS
2. Use the first-stage F-statistic to diagnose instrument strength
3. Understand why standard 2SLS confidence intervals are misleading with weak instruments
4. Implement Anderson-Rubin confidence intervals (weak-instrument robust)
5. Evaluate the tradeoff between bias and valid inference across instrument strengths

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2002)  # Stock, Wright & Yogo (2002)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Libraries loaded.")

## 1. Simulate Under Varying Instrument Strength

We vary the first stage coefficient (instrument strength) and observe how the IV estimator behaves. The true causal effect is held constant at 0.5.

In [ ]:
TRUE_EFFECT = 0.5   # True causal effect of X on Y
N = 1000
N_SIMS = 500       # simulations per scenario

def simulate_iv(n, first_stage_coef, n_sims, true_effect=0.5, seed_base=0):
    """
    Simulate IV estimation with varying instrument strength.
    Returns OLS and IV estimates across n_sims replications.
    """
    ols_ests = []
    iv_ests = []
    f_stats = []
    
    for sim in range(n_sims):
        np.random.seed(seed_base + sim)
        
        # Correlated errors (source of endogeneity)
        rho = 0.7  # correlation between X and u
        cov_matrix = np.array([[1, rho], [rho, 1]])
        errors = np.random.multivariate_normal([0, 0], cov_matrix, n)
        v = errors[:, 0]  # X equation error
        u = errors[:, 1]  # Y equation error
        
        # Instrument (exogenous)
        z = np.random.normal(0, 1, n)
        
        # X: endogenous (correlated with u via v)
        x = first_stage_coef * z + v
        
        # Y: outcome
        y = true_effect * x + u
        
        # OLS estimate
        ols = smf.ols('y ~ x', data=pd.DataFrame({'y': y, 'x': x})).fit()
        ols_ests.append(ols.params['x'])
        
        # First stage
        fs = smf.ols('x ~ z', data=pd.DataFrame({'x': x, 'z': z})).fit()
        f_stats.append(fs.fvalue)
        x_hat = fs.fittedvalues
        
        # 2SLS (manual, correct for illustration)
        iv_coef = np.cov(z, y)[0,1] / np.cov(z, x)[0,1]  # Wald formula
        iv_ests.append(iv_coef)
    
    return ols_ests, iv_ests, f_stats


# Simulate across different first-stage coefficients
first_stage_strengths = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
results = {}

print("Simulating IV under varying instrument strength...")
for pi in first_stage_strengths:
    ols, iv, f_stats = simulate_iv(N, pi, N_SIMS, TRUE_EFFECT)
    results[pi] = {
        'ols': np.array(ols),
        'iv': np.array(iv),
        'f_stats': np.array(f_stats),
        'mean_f': np.mean(f_stats),
    }
    print(f"  π = {pi:.2f}: mean F = {np.mean(f_stats):.1f}, IV mean = {np.mean(iv):.3f}, IV std = {np.std(iv):.3f}")

## 2. Bias Analysis: How IV Bias Tracks OLS Bias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Distribution of estimates for each strength
ax = axes[0]
colors = plt.cm.Blues(np.linspace(0.3, 1.0, len(first_stage_strengths)))

for i, pi in enumerate(first_stage_strengths):
    iv_est = results[pi]['iv']
    f_mean = results[pi]['mean_f']
    # Clip extreme outliers for display
    iv_clipped = iv_est[np.abs(iv_est - TRUE_EFFECT) < 5]
    ax.hist(iv_clipped, bins=40, alpha=0.5, density=True, color=colors[i],
            label=f'π={pi:.2f} (F≈{f_mean:.0f})')

ax.axvline(TRUE_EFFECT, color='red', linewidth=2.5, linestyle='--', label=f'True effect = {TRUE_EFFECT}')
ax.axvline(np.mean(results[first_stage_strengths[0]]['ols']), color='gray',
           linewidth=2, linestyle=':', label='OLS mean')
ax.set_xlabel('IV Estimate')
ax.set_ylabel('Density')
ax.set_title(f'IV Estimate Distribution by Instrument Strength\n(N={N}, {N_SIMS} simulations each)')
ax.legend(fontsize=8)
ax.set_xlim(-1, 3)

# Right: Bias and variance vs F-statistic
ax2 = axes[1]
mean_f_stats = [results[pi]['mean_f'] for pi in first_stage_strengths]
iv_biases = [np.mean(results[pi]['iv']) - TRUE_EFFECT for pi in first_stage_strengths]
iv_stds = [np.std(results[pi]['iv'][np.abs(results[pi]['iv'] - TRUE_EFFECT) < 5])
           for pi in first_stage_strengths]
ols_bias = np.mean(results[first_stage_strengths[0]]['ols']) - TRUE_EFFECT

ax2.plot(mean_f_stats, np.abs(iv_biases), 'o-', color='steelblue',
         linewidth=2, markersize=8, label='|IV bias|')
ax2.plot(mean_f_stats, iv_stds, 's--', color='darkorange',
         linewidth=2, markersize=8, label='IV std dev')
ax2.axhline(np.abs(ols_bias), color='gray', linestyle=':', linewidth=2,
            label=f'|OLS bias| = {np.abs(ols_bias):.2f}')
ax2.axvline(10, color='red', linestyle='--', alpha=0.7, label='F = 10 threshold')

ax2.set_xlabel('Mean First Stage F-statistic')
ax2.set_ylabel('Absolute Bias / Std Dev')
ax2.set_title('IV Bias and Variance vs Instrument Strength')
ax2.legend(fontsize=9)
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

print("\nSummary table:")
print(f"{'π':>6} {'Mean F':>8} {'IV Bias':>10} {'OLS Bias':>10} {'IV as % of OLS bias':>20}")
print("-" * 60)
for pi in first_stage_strengths:
    iv_b = np.mean(results[pi]['iv']) - TRUE_EFFECT
    ols_b = np.mean(results[pi]['ols']) - TRUE_EFFECT
    pct = 100 * iv_b / ols_b if ols_b != 0 else np.inf
    print(f"{pi:>6.2f} {results[pi]['mean_f']:>8.1f} {iv_b:>+10.4f} {ols_b:>+10.4f} {pct:>19.1f}%")

## 3. Confidence Interval Coverage: When 2SLS CIs Are Misleading

In [ ]:
def check_ci_coverage(n, first_stage_coef, n_sims=500, true_effect=0.5):
    """
    Check whether 95% CI from 2SLS contains the true value 95% of the time.
    """
    coverage = []
    
    for sim in range(n_sims):
        np.random.seed(sim)
        rho = 0.7
        cov_m = np.array([[1, rho], [rho, 1]])
        err = np.random.multivariate_normal([0, 0], cov_m, n)
        v, u = err[:, 0], err[:, 1]
        z = np.random.normal(0, 1, n)
        x = first_stage_coef * z + v
        y = true_effect * x + u
        
        data = pd.DataFrame({'y': y, 'x': x, 'z': z})
        
        # 2SLS using Wald + bootstrap SE approximation
        iv_est = np.cov(z, y)[0,1] / np.cov(z, x)[0,1]
        
        # Standard 2SLS SE (asymptotic formula)
        x_hat = smf.ols('x ~ z', data=data).fit().fittedvalues
        residuals = y - iv_est * x
        sigma2 = np.var(residuals)
        iv_var = sigma2 / (np.sum((x_hat - x_hat.mean())**2))
        iv_se = np.sqrt(iv_var)
        
        ci_lo = iv_est - 1.96 * iv_se
        ci_hi = iv_est + 1.96 * iv_se
        coverage.append(ci_lo <= true_effect <= ci_hi)
    
    return np.mean(coverage)

print("95% CI Coverage Rate for 2SLS (should be 0.95):")
print("=" * 45)
print(f"{'π':>6} {'Mean F':>8} {'Coverage':>10}")
print("-" * 45)
for pi in [0.05, 0.1, 0.2, 0.5, 1.0, 2.0]:
    cov = check_ci_coverage(N, pi, n_sims=300)
    f_approx = results[pi]['mean_f']
    flag = " ← INADEQUATE" if cov < 0.90 else ""
    print(f"{pi:>6.2f} {f_approx:>8.1f} {cov:>10.3f}{flag}")

print("-" * 45)
print("With weak instruments: CI coverage is far below nominal 95%")
print("This means published results with weak instruments are misleading")

## 4. Practical Recommendations

In [ ]:
print("=" * 60)
print("INSTRUMENT STRENGTH ASSESSMENT GUIDE")
print("=" * 60)

guidelines = [
    ("F < 5",   "VERY WEAK",   "IV is severely biased. Do not use. Report OLS with caveat."),
    ("5 ≤ F < 10", "WEAK",    "IV is notably biased. Use AR intervals if you must proceed."),
    ("10 ≤ F < 20", "MODERATE", "IV bias at most 10% of OLS. Standard 2SLS acceptable."),
    ("20 ≤ F < 100", "STRONG", "Conventional IV results reliable."),
    ("F ≥ 100",  "VERY STRONG", "Nearly unbiased IV. Standard results fully reliable."),
]

for f_range, category, recommendation in guidelines:
    print(f"\n{f_range:<12} [{category}]")
    print(f"  → {recommendation}")

print("\n" + "=" * 60)
print("WHAT TO DO WITH WEAK INSTRUMENTS")
print("=" * 60)
print("""
Option 1: Find a stronger instrument
  - Often the best solution
  - More exogenous variation, stronger first stage

Option 2: Add instruments
  - Multiple weak instruments can be collectively strong
  - Risk: if any violate exclusion, you have compound bias

Option 3: Use Anderson-Rubin confidence intervals
  - Valid regardless of instrument strength
  - CI may be wide or even unbounded — that's honest
  - Available in linearmodels, ivmodels packages

Option 4: LIML (Limited Information Maximum Likelihood)
  - Less biased than 2SLS with weak instruments
  - Fuller's k (k=1) is a good default
  - Available in linearmodels
""")

## Self-Check

1. Set `rho = 0.0` (no endogeneity). Does the weak instrument problem disappear? What does IV estimate in this case?

2. Try `N = 10000` (large sample) with a weak instrument (`pi = 0.1`). Does the large sample fix the weak instrument problem? (Hint: look at whether bias decreases with larger N.)

3. Implement the Anderson-Rubin test manually: grid search over possible values of `beta` and keep those where the regression `y - beta*x ~ z` gives a non-significant coefficient on `z`. Compare this CI to the standard 2SLS CI.

4. Try `rho = 0.95` (very strong endogeneity) with `pi = 0.5` (adequate instrument). How does the IV bias compare to the OLS bias in this case?

---
**Next:** `03_combined_designs.ipynb` — Combining causal designs